# ACCESS-AIS3 -- SSA grounded-ice friction inversion

## Imports & helper functions

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os


def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)


def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds


def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Configure options

In [ ]:
## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'


all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
]

# Define steps to run (this notebook is scoped to this group)
steps = ['ssa_friction_forward_check', 'ssa_friction_forward_check_budd', 'ssa_friction_inv_sensit', 'ssa_friction_inv_lcurve', 'ssa_friction_inv_reg_lcurve']

## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

## Initialise data catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## SSA Forward Convergence Check - Friction Domain (no inversion)

In [ ]:
## ------------------------------------
## SSA Forward Convergence Check - Friction Domain (no inversion)
## ------------------------------------
# Diagnostic: reproduce the friction inversion domain/BCs exactly, but disable the
# inversion and impose a uniform, physically-reasonable friction C. Solve a SINGLE
# forward stress balance and inspect whether the non-linear (viscosity) iteration
# converges. If the forward solve diverges here, the friction inversion cannot work
# regardless of the cost coefficients -- fix the forward model (BCs) first.
if 'ssa_friction_forward_check' in steps:

    print("-------------------------------------------------------------")
    print(f" SSA FORWARD CONVERGENCE CHECK - FRICTION DOMAIN"            )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results ({rheology_lcurve_run})...")
    mds = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/{rheology_lcurve_run}/{rheology_lcurve_run}.nc')

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[mds.mesh.extractedvertices - 1] = mds.results.StressbalanceSolution.MaterialsRheologyBbar # Note: -1 for zero-based indexing

    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    # Fix negative effective pressure (identical to the inversion setup)
    N = md.friction.effective_pressure.copy()
    N[N < 0] = 0
    md.friction.effective_pressure = N
    md.friction.effective_pressure_limit = 0.07  # match the N floor set in param

    print(f"-- Extracting friction-inversion domain (floating/grounded ice-front BC)...")
    mds = extract_friction_inversion_domain(md)

    print(f"-- Disabling inversion (forward solve only)...")
    mds.inversion.iscontrol = 0
    mds.verbose.solution = 1  # print non-linear convergence to the log

    # Identify purely-floating elements once (these get ~0 friction regardless of the sweep value)
    ocean_elements = mds.mask.ocean_levelset[mds.mesh.elements - 1] # -1 for zero-based indexing
    pos_e = np.where(np.min(ocean_elements, axis=1) < 0)[0]
    flags = np.zeros(mds.mesh.numberofvertices, dtype=bool)
    flags[mds.mesh.elements[pos_e, :] - 1] = True # -1 for zero-based indexing

    mds.transient = pyissm.model.classes.transient.deactivate_all(mds.transient)

    # Same non-linear solver tolerances as the inversion forward solves
    mds.stressbalance.restol = 0.01
    mds.stressbalance.reltol = 0.1
    mds.stressbalance.abstol = np.nan
    mds.settings.solver_residue_threshold = 1e-3

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # Sweep a few uniform grounded-ice C values. Basal drag scales like C^2, so this brackets
    # the coefficient magnitude needed to tame the ice by orders of magnitude. If a large enough
    # C brings velocities into the physical range, the fix is to raise the inversion bounds /
    # rescale C; if even the largest value blows up, the friction is not coupling (a real bug).
    forward_check_C_values = [5000]

    for Cval in forward_check_C_values:
        run_name = f'AIS3_friction_forward_check_C{Cval}'
        print(f"\n-- Uniform grounded C = {Cval}  ({run_name}) --")

        mds.friction.C = np.full(mds.mesh.numberofvertices, float(Cval))
        mds.friction.C[flags] = 0.05
        mds.miscellaneous.name = run_name

        if save:
            mdi = pyissm.model.execute.solve(mds, 'Stressbalance', load_only = True, runtime_name = False)
            vel = np.asarray(mdi.results.StressbalanceSolution.Vel).ravel()
            vel_obs = np.asarray(mdi.inversion.vel_obs).ravel()
            grounded = (mdi.mask.ice_levelset < 0) & (mdi.mask.ocean_levelset > 0)
            print(f"   obs max={np.nanmax(vel_obs):.0f} | mod max={np.nanmax(vel):.0f} "
                  f"med={np.nanmedian(vel):.0f} | grounded med={np.nanmedian(vel[grounded]):.0f} "
                  f"| nodes>1e4: {(vel > 1e4).sum()}/{vel.size} ({100*(vel > 1e4).mean():.1f}%)")
        else:
            pyissm.model.execute.solve(mds, 'Stressbalance', load_only = False, runtime_name = False)

## SSA Forward Convergence Check - Friction Domain (BUDD law experiment)

In [ ]:
# Same friction domain and BCs as ssa_friction_forward_check, but replaces the Schoof
# (regularized-Coulomb) law with the Budd/Weertman power law, which has NO Coulomb ceiling
# (tau_b = coefficient^2 * Neff^r * |u|^(s-1) * u, r=q/p, s=1/p). Removing the Cmax*N cap
# should eliminate the unfittable Coulomb-failure nodes and the inversion overshoot. Sweeps a
# uniform coefficient to find the magnitude that gives physical velocities; if it does, Budd
# is worth adopting for the friction inversion. Self-contained -- does not touch param or the
# Schoof blocks, so both laws remain available for comparison.
if 'ssa_friction_forward_check_budd' in steps:

    print("-------------------------------------------------------------")
    print(f" SSA FORWARD CHECK (BUDD LAW) - FRICTION DOMAIN"             )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results ({rheology_lcurve_run})...")
    mdr = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/{rheology_lcurve_run}/{rheology_lcurve_run}.nc')
    md.materials.rheology_B[mdr.mesh.extractedvertices - 1] = mdr.results.StressbalanceSolution.MaterialsRheologyBbar

    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    # Fix negative effective pressure (same floored N as the Schoof setup)
    N = md.friction.effective_pressure.copy()
    N[N < 0] = 0
    md.friction.effective_pressure = N
    md.friction.effective_pressure_limit = 0.07

    print(f"-- Extracting friction-inversion domain (floating/grounded ice-front BC)...")
    mds = extract_friction_inversion_domain(md)

    print(f"-- Switching friction law to Budd (default class, no Coulomb ceiling)...")
    # default(other) inherits effective_pressure / limit / coupling from the Schoof friction.
    # p, q are per-element: s = 1/p, r = q/p. p=3, q=3 -> tau_b ~ N*|u|^(1/3) (Budd, linear
    # N-dependence, m=3 sliding). Tune p/q as needed (q=0 -> Weertman, no N-dependence).
    mds.friction = pyissm.model.classes.friction.default(mds.friction)
    mds.friction.p = np.full(mds.mesh.numberofelements, 3.0)
    mds.friction.q = np.full(mds.mesh.numberofelements, 3.0)
    mds.friction.coupling = 3  # use the provided (floored) effective_pressure

    print(f"-- Disabling inversion (forward solve only)...")
    mds.inversion.iscontrol = 0
    mds.verbose.solution = 1

    # Purely-floating elements -> ~0 friction
    ocean_elements = mds.mask.ocean_levelset[mds.mesh.elements - 1]
    pos_e = np.where(np.min(ocean_elements, axis=1) < 0)[0]
    flags = np.zeros(mds.mesh.numberofvertices, dtype=bool)
    flags[mds.mesh.elements[pos_e, :] - 1] = True

    mds.transient = pyissm.model.classes.transient.deactivate_all(mds.transient)
    mds.stressbalance.restol = 0.01
    mds.stressbalance.reltol = 0.1
    mds.stressbalance.abstol = np.nan
    mds.settings.solver_residue_threshold = 1e-3

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # Sweep uniform Budd coefficient (magnitude unknown a priori; brackets a few decades).
    budd_coeff_values = [1, 10, 100, 1000]

    for cval in budd_coeff_values:
        run_name = f'AIS3_friction_forward_check_budd_{cval}'
        print(f"\n-- Uniform Budd coefficient = {cval}  ({run_name}) --")
        mds.friction.coefficient = np.full(mds.mesh.numberofvertices, float(cval))
        mds.friction.coefficient[flags] = 0.05
        mds.miscellaneous.name = run_name

        if save:
            mdi = pyissm.model.execute.solve(mds, 'Stressbalance', load_only = True, runtime_name = False)
            vel = np.asarray(mdi.results.StressbalanceSolution.Vel).ravel()
            vel_obs = np.asarray(mdi.inversion.vel_obs).ravel()
            grounded = (mdi.mask.ice_levelset < 0) & (mdi.mask.ocean_levelset > 0)
            print(f"   obs max={np.nanmax(vel_obs):.0f} | mod max={np.nanmax(vel):.0f} "
                  f"med={np.nanmedian(vel):.0f} | grounded med={np.nanmedian(vel[grounded]):.0f} "
                  f"| nodes>1e4: {(vel > 1e4).sum()}/{vel.size} ({100*(vel > 1e4).mean():.1f}%)")
        else:
            pyissm.model.execute.solve(mds, 'Stressbalance', load_only = False, runtime_name = False)

## SSA Friction Inversion Sensitivity - Grounded Ice

In [ ]:
if 'ssa_friction_inv_sensit' in steps:

    print("-------------------------------------------------------------")
    print(f" SSA FRICTION INVERSION SENSITIVITY - GROUNDED ICE"          )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results...")
    mds = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17/run_004_1_10_1e-17.nc')

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[mds.mesh.extractedvertices - 1] = mds.results.StressbalanceSolution.MaterialsRheologyBbar # Note: -1 for zero-based indexing

    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    # Fix negative effective pressure
    # TODO: Update this in param
    N = md.friction.effective_pressure.copy()
    N[N < 0] = 0
    md.friction.effective_pressure = N
    md.friction.effective_pressure_limit = 0.07  # match the N floor set in param

    print(f"-- Extracting friction-inversion domain (floating/grounded ice-front BC)...")
    mds = extract_friction_inversion_domain(md)

    print(f"-- Overriding friction.coupling = {friction_coupling}...")
    mds.friction.coupling = friction_coupling

    print(f"-- Defining inversion parameters...")
    mds.inversion = pyissm.model.classes.inversion.m1qn3(mds.inversion)
    fric_control, fric_field, fric_min, fric_max = friction_law_info(mds)  # Schoof or Budd
    mds.inversion.control_parameters = [fric_control]
    mds.inversion.min_parameters = np.full(mds.mesh.numberofvertices, fric_min)
    mds.inversion.max_parameters = np.full(mds.mesh.numberofvertices, fric_max)  # bounds fixed above (friction_law_info)
    # The "converges in ~15 steps" premise here was measured against the C_init=10 dead-zone bug
    # (see ais_0.1_param.py): with the model stuck at zero velocity, m1qn3 had nothing to do and
    # "converged" immediately by never moving. VALIDATED (grounded RMSE 98.9, `friconly_nfix`):
    # the real fit needs 192 m1qn3 iterations to reach dxmin, with the cost still improving as
    # late as iteration ~150. 30/50 would cut this run off before the fit has developed at all.
    # The OOM history this comment used to cite is addressed above (cluster.np/memory) -- 100GB
    # was under the ~130GB this mesh needs even at 32 ranks, not a consequence of maxsteps itself.
    mds.inversion.maxsteps = 500
    mds.inversion.maxiter = 500

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # No friction on PURELY floating ice elements
    # TODO: Initialise the friction field as a float in param to prevent the need to convert it here to avoid >0 consistency issue
    ocean_elements = mds.mask.ocean_levelset[mds.mesh.elements - 1] # -1 for zero-based indexing
    pos_e = np.where(np.min(ocean_elements, axis=1) < 0)[0]
    flags = np.zeros(mds.mesh.numberofvertices, dtype=bool)
    flags[mds.mesh.elements[pos_e, :] - 1] = True # -1 for zero-based indexing
    _fld = getattr(mds.friction, fric_field).astype(float)
    _fld[flags] = 0.05
    setattr(mds.friction, fric_field, _fld)
    mds.inversion.min_parameters[flags] = 0.0
    mds.inversion.max_parameters[flags] = 0.0

    mds.transient = pyissm.model.classes.transient.deactivate_all(mds.transient)

    mds.stressbalance.restol = 0.01
    mds.stressbalance.reltol = 0.1
    mds.stressbalance.abstol = np.nan
    mds.settings.solver_residue_threshold = 1e-3

    print(f"-- Setting-up coefficient grid...")
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [0.1, 1, 10, 100, 1000],
         103: [0.1, 1, 10, 100, 1000]})

    # TODO: Fill NaN obs vel with 0, not NN so that these regions can be excluded. Update in Param
    # TODO: Exclude ice-front from inversion as well -- mds.inversion.cost_functions_coefficients(iceFront, 1:2) = 0
    # NOTE: An a-priori Coulomb-failure cost mask (tau_d = rho*g*H*|grad(s)| > Cmax*N) was tried
    # here to drop unfittable nodes, but the static driving-stress proxy did not match the actual
    # dynamic blowup cells (bulk fit unchanged, and it pushed some runs into overshoot). Reverted.
    # VALIDATED FIX: restrict the 101/103 velocity misfit to GROUNDED observed ice. This control
    # (FrictionCoefficient) is frozen at ~0 on floating ice, so shelf model-obs mismatch could
    # never be corrected where it arises -- it pushed through the grounding line and was absorbed
    # by grounded friction instead (in `friconly_nfix`, floating vertices were 16% of the observed
    # misfit before this fix). Shelves stay in the domain and still buttress; they just stop
    # driving the grounded control.
    print(f"-- Defining mask to exclude 0 velocity and restrict to grounded ice...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ocean_levelset >= 0)

    if save:
        print(f"-- Loading inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_friction_inv_sensit',
            run = False,
            load_only = True,
            global_mask = mask)

        print(f"-- Processing inversion parameter sensitivity...")
        diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir=f'{model_dir}/AIS3_ssa_friction_inv_sensit/')

        diagnostics_norm = pyissm.inversion.sensitivity.normalize_diagnostics(diagnostics,
                                                                             columns = ['vel_rmse', 'mean_gradient_magnitude'])
        diagnostics_norm['overall'] = diagnostics_norm['vel_rmse_norm'] + diagnostics_norm['mean_gradient_magnitude_norm']

        fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize = (15, 8), constrained_layout = True)
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax1, value = 'vel_rmse')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax2, value = 'ratio_101_103')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax3, value = 'mean_gradient_magnitude')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax4, value = 'cost_total')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax5, value = 'positive_residual_fraction')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics_norm, x = 'cf101', y = 'cf103', ax = ax6, value = 'overall')
        plt.savefig(f'{model_dir}/AIS3_ssa_friction_inv_sensit/diagnostic_heatmaps.png')

        best_row = diagnostics_norm.loc[diagnostics_norm['overall'].idxmax()]
        print(f"The best run_id is: {best_row['run_id']}. This uses the following coefficient values:")
        print(best_row.filter(regex=r'^cf'))

    else:
        print(f"-- Running inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_friction_inv_sensit',
            run = True,
            load_only = False,
            global_mask = mask)

## SSA Friction Inversion L-Curve - Grounded Ice

In [ ]:
if 'ssa_friction_inv_lcurve' in steps:

    print("-------------------------------------------------------------")
    print(f" SSA FRICTION INVERSION - UNREGULARISED BASELINE (p=q=1)"    )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results (execution_newB_rheology, see load_shelf_rheology_B docstring)...")
    _ev, _Bshelf = load_shelf_rheology_B()

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[_ev - 1] = _Bshelf # Note: -1 for zero-based indexing

    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print(f"-- Flooring thin ice at 100m (numerical stability), preserving observed surface...")
    # Production param.py already floors thickness at 10m (ais_0.1_param.py:62), with N
    # consistently floored against that same 10m value at param time -- internally
    # consistent, but 10m proved numerically fragile during this project's own testing (thin,
    # steep-terrain vertices at that thickness were the source of repeated velocity
    # runaways). VALIDATED fix: raise the floor to 100m, but absorb the extra thickness into
    # the BASE only, leaving the OBSERVED surface (hence grad(surface), hence driving
    # stress) untouched. An earlier attempt that instead rebuilt surface as bed+H moved the
    # surface by up to +90m and fabricated driving stress (worst case RMSE 269.6, Transantarctic
    # Mountains). Also re-floor N against the NEW 100m thickness: N was already floored once in
    # param.py, but against the stale 10m value -- left unrefloored, thin-ice vertices carry
    # ~10x too little basal drag for their now-larger overburden, with no C value able to
    # compensate (this was the exact mechanism behind a 14.56x-too-fast runaway group found
    # earlier this project).
    _ri = md.materials.rho_ice; _rw = md.materials.rho_water
    _H = np.asarray(md.geometry.thickness).ravel().copy()
    _ol = np.asarray(md.mask.ocean_levelset).ravel()
    _surf0 = np.asarray(md.geometry.surface).ravel().copy()
    _nfl = int((_H < 100).sum())
    _H = np.maximum(_H, 100.0)
    _flt = _ol < 0
    _base = np.empty_like(_H); _surf = np.empty_like(_H)
    _base[_flt] = -_H[_flt] * _ri / _rw; _surf[_flt] = _H[_flt] * (1.0 - _ri / _rw)
    _surf[~_flt] = _surf0[~_flt]
    _base[~_flt] = _surf0[~_flt] - _H[~_flt]
    md.geometry.thickness = _H; md.geometry.base = _base; md.geometry.surface = _surf
    print(f"   {_nfl} verts floored to 100m; max surface change on grounded ice = "
          f"{np.nanmax(np.abs(_surf[~_flt] - _surf0[~_flt])):.3g} m (should be 0)")

    print(f"-- Re-flooring effective pressure against the updated (100m-floored) thickness...")
    _lim = 0.07
    N = md.friction.effective_pressure.copy()
    N[N < 0] = 0
    _Nfloor = _lim * md.materials.rho_ice * md.constants.g * _H
    _nbad = int(np.sum(N < _Nfloor * 0.999))
    N = np.maximum(N, _Nfloor)
    print(f"   {_nbad} verts raised to the updated N floor")
    md.friction.effective_pressure = N
    md.friction.effective_pressure_limit = _lim

    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Extracting friction-inversion domain (floating/grounded ice-front BC)...")
    mds = extract_friction_inversion_domain(md)

    print(f"-- Overriding friction.coupling = {friction_coupling}...")
    mds.friction.coupling = friction_coupling

    print(f"-- Re-flooring N on the extracted domain (sanity check)...")
    _Hs = np.asarray(mds.geometry.thickness).ravel()
    _Ns = mds.friction.effective_pressure.copy(); _Ns[_Ns < 0] = 0
    _Nfs = 0.07 * mds.materials.rho_ice * mds.constants.g * _Hs
    mds.friction.effective_pressure = np.maximum(_Ns, _Nfs)
    mds.friction.effective_pressure_limit = 0.07
    _gr_check = (np.asarray(mds.mask.ice_levelset).ravel() < 0) & (np.asarray(mds.mask.ocean_levelset).ravel() > 0)
    _frac = mds.friction.effective_pressure / np.maximum(mds.materials.rho_ice * mds.constants.g * _Hs, 1e-9)
    assert np.nanmin(_frac[_gr_check]) >= 0.0699, "N still below floor after re-flooring"

    print(f"-- Defining inversion parameters...")
    mds.inversion = pyissm.model.classes.inversion.m1qn3(mds.inversion)
    fric_control, fric_field, fric_min, fric_max = friction_law_info(mds)  # Schoof or Budd
    mds.inversion.control_parameters = [fric_control]
    mds.inversion.min_parameters = np.full(mds.mesh.numberofvertices, fric_min)
    mds.inversion.max_parameters = np.full(mds.mesh.numberofvertices, fric_max)  # bounds fixed above (friction_law_info)
    # See the updated note in ssa_friction_inv_sensit: the "keep budget small" premise was
    # measured against the C_init=10 dead-zone bug and the OOM history is addressed by the
    # cluster.np/memory fix above, not by capping maxsteps. VALIDATED: 500/500 (p=q=1 converges
    # to grounded RMSE 61.4 at 198 iterations, on dxmin -- see friction_inv_gttol below for why
    # that's the criterion that actually has to fire).
    mds.inversion.maxsteps = 500
    mds.inversion.maxiter = 500
    # See friction_inv_gttol definition above: without this, m1qn3's default gttol=1e-4 stops
    # every p=q=1 run by iteration ~16 on a false "converged" signal, before the fit has done
    # anything -- this is the single fix that made p=q=1 (and its RMSE win over p=q=3) visible
    # at all; every earlier p=1 attempt silently never tested it.
    mds.inversion.gttol = friction_inv_gttol

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # No friction on floating ice vertices. BUGFIX: was previously element-based
    # (np.min(ocean_elements, axis=1) < 0, i.e. ANY vertex of the element floating), which --
    # despite the "PURELY floating ice elements" comment -- actually zeroed friction bounds on
    # GROUNDED vertices merely adjacent to a floating one along the whole grounding line. The
    # validated script (p1q1_reg_sweep2.py, which produced the actual RMSE 60.4/61.4 result)
    # never used element logic at all -- plain per-vertex floating flags, matching here.
    flags = np.asarray(mds.mask.ocean_levelset).ravel() < 0
    _fld = getattr(mds.friction, fric_field).astype(float)
    _fld[flags] = 0.05
    setattr(mds.friction, fric_field, _fld)
    mds.inversion.min_parameters[flags] = 0.0
    mds.inversion.max_parameters[flags] = 0.0

    mds.transient = pyissm.model.classes.transient.deactivate_all(mds.transient)

    mds.stressbalance.restol = 0.01
    mds.stressbalance.reltol = 0.1
    mds.stressbalance.abstol = np.nan
    mds.settings.solver_residue_threshold = 1e-3

    print(f"-- Setting-up coefficient grid (single unregularised baseline run)...")
    # This step is now ONLY the unregularised p=q=1 baseline (cf501 negligible, effectively
    # off) -- its sole purpose is to be the warm-start source for ssa_friction_inv_reg_lcurve
    # below, which does the actual cf501 (DragCoefficientAbsGradient) L-curve sweep. Folding a
    # 501 grid directly into this step (as the old p=q=3-era code did) doesn't reproduce the
    # validated methodology: each grid point there cold-starts from C_init=1.8 independently,
    # never warm-started from a converged state, which is a different (and unvalidated)
    # experiment. See ssa_friction_inv_reg_lcurve for the real sweep and the p=q=3 -> p=q=1
    # 501-scale note (C's units/magnitude differ completely between the two laws, so the old
    # p=3-tuned 1e-3..3e-1 range is meaningless here -- validated p=1 range is ~1e-4..1e-2).
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [friction_cf101],
         103: [friction_cf103],
         501: [1e-8]})

    # VALIDATED FIX: restrict 101/103 to grounded observed ice (see the matching fix and
    # rationale in ssa_friction_inv_sensit above).
    print(f"-- Defining mask to exclude 0 velocity and restrict to grounded ice...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ocean_levelset >= 0)

    # VALIDATED FIX: exclude grounding-line-adjacent elements from the 501 regularisation mask.
    # C steps from its grounded value to ~0 across the grounding line (floating C is pinned to
    # a near-zero bound), and coeff_masks are per-vertex weights ISSM integrates element-wise,
    # so a plain grounded mask still weights elements straddling that discontinuity -- and since
    # floating C cannot move, that part of the penalty is irreducible. See section 3.1.
    grounded_mask = np.asarray(mds.mask.ocean_levelset) >= 0
    _elx = np.asarray(mds.mesh.elements).astype(int) - 1
    _float_v = np.asarray(mds.mask.ocean_levelset).ravel() < 0
    _touch_f = _float_v[_elx].any(axis = 1)
    _gladj = np.zeros(mds.mesh.numberofvertices, dtype = bool)
    _gladj[_elx[_touch_f].ravel()] = True
    reg_mask = grounded_mask & ~_gladj

    if save:
        # Single-point baseline, not a sweep -- no L-curve to plot here (see
        # ssa_friction_inv_reg_lcurve for the actual cf501 L-curve). Just load and report
        # grounded RMSE against the validated reference (61.4).
        print(f"-- Loading baseline friction inversion result...")
        mdb = pyissm.model.io.load_model(
            f'{model_dir}/AIS3_ssa_friction_inv_lcurve/{friction_baseline_run}/{friction_baseline_run}.nc')
        vel = np.asarray(mdb.results.StressbalanceSolution.Vel).ravel()
        vo = np.asarray(mdb.inversion.vel_obs).ravel()
        gr = (np.asarray(mdb.mask.ice_levelset).ravel() < 0) & (np.asarray(mdb.mask.ocean_levelset).ravel() > 0)
        rmse = np.sqrt(np.nanmean((vel[gr] - vo[gr]) ** 2))
        print(f"   grounded RMSE = {rmse:.1f} m/yr (validated reference: 61.4)")

    else:
        print(f"-- Running unregularised p=q=1 baseline inversion...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_friction_inv_lcurve',
            run = True,
            load_only = False,
            coeff_masks = {101: mask,
                           103: mask,
                           501: reg_mask})

## Friction regularisation L-curve, warm-started from the unregularised baseline

In [ ]:
if 'ssa_friction_inv_reg_lcurve' in steps:

    print("-------------------------------------------------------------")
    print(f" SSA FRICTION INVERSION L-CURVE - cf501 REGULARISATION (p=q=1)")
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results (execution_newB_rheology, see load_shelf_rheology_B docstring)...")
    _ev, _Bshelf = load_shelf_rheology_B()

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[_ev - 1] = _Bshelf

    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print(f"-- Flooring thin ice at 100m (numerical stability), preserving observed surface...")
    # Same fix as ssa_friction_inv_lcurve above -- must be applied identically here so this
    # step's geometry matches what friction_baseline_run was actually solved against; a
    # mismatch would mean the warm-started C field below gets paired with different driving
    # stress than it was tuned for. See ssa_friction_inv_lcurve for the full rationale.
    _ri = md.materials.rho_ice; _rw = md.materials.rho_water
    _H = np.asarray(md.geometry.thickness).ravel().copy()
    _ol = np.asarray(md.mask.ocean_levelset).ravel()
    _surf0 = np.asarray(md.geometry.surface).ravel().copy()
    _H = np.maximum(_H, 100.0)
    _flt = _ol < 0
    _base = np.empty_like(_H); _surf = np.empty_like(_H)
    _base[_flt] = -_H[_flt] * _ri / _rw; _surf[_flt] = _H[_flt] * (1.0 - _ri / _rw)
    _surf[~_flt] = _surf0[~_flt]
    _base[~_flt] = _surf0[~_flt] - _H[~_flt]
    md.geometry.thickness = _H; md.geometry.base = _base; md.geometry.surface = _surf

    _lim = 0.07
    N = md.friction.effective_pressure.copy()
    N[N < 0] = 0
    _Nfloor = _lim * md.materials.rho_ice * md.constants.g * _H
    N = np.maximum(N, _Nfloor)
    md.friction.effective_pressure = N
    md.friction.effective_pressure_limit = _lim

    print(f"-- Define general control parameters...")
    # BUGFIX: this step was missing this line entirely -- md.inversion.iscontrol defaults to 0
    # on a fresh AIS3_param.nc load, and the m1qn3(mds.inversion) reconstruction below only
    # INHERITS iscontrol from its input (pyissm/model/classes/inversion.py: m1qn3.__init__ ->
    # super().__init__(other)), it doesn't set it. Without this, the step silently runs a
    # single forward stress-balance solve ("computing new velocity" in the outlog) instead of
    # an m1qn3 control inversion -- confirmed by direct A/B test (see docs/inversion_worklog.md):
    # models/AIS3_ssa_friction_inv_reg_lcurve/ doesn't exist on disk, meaning this step has
    # never actually been run since the p=1 restructuring; every "RMSE 60.4, cf501=0.0001"
    # result referenced elsewhere in this project came from an ad-hoc scratchpad script
    # (p1q1_reg_sweep2.py) that set this correctly, not from this production step.
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Extracting friction-inversion domain (floating/grounded ice-front BC)...")
    mds = extract_friction_inversion_domain(md)

    print(f"-- Overriding friction.coupling = {friction_coupling}...")
    mds.friction.coupling = friction_coupling

    print(f"-- Re-flooring N on the extracted domain (sanity check)...")
    _Hs = np.asarray(mds.geometry.thickness).ravel()
    _Ns = mds.friction.effective_pressure.copy(); _Ns[_Ns < 0] = 0
    _Nfs = 0.07 * mds.materials.rho_ice * mds.constants.g * _Hs
    mds.friction.effective_pressure = np.maximum(_Ns, _Nfs)
    mds.friction.effective_pressure_limit = 0.07

    if use_constrain_regions:
        print(f"-- Constraining velocity to observations inside {constrain_exp_file}...")
        # Mirrors Felicity's Inversion_Friction_Budd pattern (runme.m:624-627): inside the
        # polygon, spcvx/spcvy/spcvz are pinned to observed velocity (Dirichlet) instead of
        # being left free for the inversion, so the solver can't drive velocity away from
        # observations in the two known-unstable clusters this file encodes.
        _cpos = pyissm.tools.wrappers.ContourToNodes(
            np.asarray(mds.mesh.x).ravel(), np.asarray(mds.mesh.y).ravel(),
            constrain_exp_file, 2).astype(bool)
        _vxo = np.asarray(mds.inversion.vx_obs).ravel()
        _vyo = np.asarray(mds.inversion.vy_obs).ravel()
        mds.stressbalance.spcvx[_cpos] = _vxo[_cpos]
        mds.stressbalance.spcvy[_cpos] = _vyo[_cpos]
        mds.stressbalance.spcvz[_cpos] = 0
        print(f"   pinned {int(_cpos.sum())} vertices to observed velocity")

    print(f"-- Warm-starting C from the unregularised baseline ({friction_baseline_run})...")
    # Warm-starting (rather than cold-starting each 501 grid point from C_init=1.8, as the
    # p=q=3-era code did) is the validated methodology: the regularisation sweep only needs
    # to locally smooth an already-converged C field, not re-solve the whole continent per
    # grid point. This also matches how every successful 501 sweep in this project has
    # actually been run.
    mdb = pyissm.model.io.load_model(
        f'{model_dir}/AIS3_ssa_friction_inv_lcurve/{friction_baseline_run}/{friction_baseline_run}.nc')
    C_baseline = np.asarray(mdb.results.StressbalanceSolution.FrictionCoefficient).ravel()

    print(f"-- Defining inversion parameters...")
    mds.inversion = pyissm.model.classes.inversion.m1qn3(mds.inversion)
    fric_control, fric_field, fric_min, fric_max = friction_law_info(mds)
    mds.inversion.control_parameters = [fric_control]
    mds.inversion.min_parameters = np.full(mds.mesh.numberofvertices, fric_min)
    mds.inversion.max_parameters = np.full(mds.mesh.numberofvertices, fric_max)
    mds.inversion.maxsteps = 500
    mds.inversion.maxiter = 500
    mds.inversion.gttol = friction_inv_gttol

    setattr(mds.friction, fric_field, C_baseline.copy())

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # No friction on floating ice vertices. BUGFIX: was previously element-based
    # (np.min(ocean_elements, axis=1) < 0, i.e. ANY vertex of the element floating), which --
    # despite the "PURELY floating ice elements" comment -- actually zeroed friction bounds on
    # GROUNDED vertices merely adjacent to a floating one along the whole grounding line. The
    # validated script (p1q1_reg_sweep2.py, which produced the actual RMSE 60.4/61.4 result)
    # never used element logic at all -- plain per-vertex floating flags, matching here.
    flags = np.asarray(mds.mask.ocean_levelset).ravel() < 0
    _fld = getattr(mds.friction, fric_field).astype(float)
    _fld[flags] = 0.05
    setattr(mds.friction, fric_field, _fld)
    mds.inversion.min_parameters[flags] = 0.0
    mds.inversion.max_parameters[flags] = 0.0

    mds.transient = pyissm.model.classes.transient.deactivate_all(mds.transient)

    mds.stressbalance.restol = 0.01
    mds.stressbalance.reltol = 0.1
    mds.stressbalance.abstol = np.nan
    mds.settings.solver_residue_threshold = 1e-3

    print(f"-- Setting-up coefficient grid...")
    # VALIDATED range for p=q=1 (found by direct sweep warm-started from the RMSE-61.4
    # baseline): cf501=0.0001 drops C-field roughness from 0.82 to 0.18 (matching the p=q=3
    # baseline's own 0.17) while RMSE *improves* to 60.4 -- both axes better, not a tradeoff.
    # RMSE degrades fast past that point (70.5 at 0.0003, 307.8 by 0.016), so 0.0001 is the
    # chosen corner (`friction_lcurve_run` above), not just a swept value. This range is NOT
    # comparable to the old p=q=3 1e-3..3e-1 grid -- C's units/magnitude differ completely
    # between the two laws (u ~ C^-6 vs u ~ C^-2), so the two sweeps can't share a scale.
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [friction_cf101],
         103: [friction_cf103],
         501: [0.0001, 0.0003, 0.001, 0.002, 0.0032]})

    print(f"-- Defining mask to exclude 0 velocity and restrict to grounded ice...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ocean_levelset >= 0)

    # Exclude grounding-line-adjacent elements from the 501 regularisation mask -- see the
    # matching note in ssa_friction_inv_lcurve / docs/inversion_worklog.md section 3.1.
    grounded_mask = np.asarray(mds.mask.ocean_levelset) >= 0
    _elx = np.asarray(mds.mesh.elements).astype(int) - 1
    _float_v = np.asarray(mds.mask.ocean_levelset).ravel() < 0
    _touch_f = _float_v[_elx].any(axis = 1)
    _gladj = np.zeros(mds.mesh.numberofvertices, dtype = bool)
    _gladj[_elx[_touch_f].ravel()] = True
    reg_mask = grounded_mask & ~_gladj

    if save:
        print(f"-- Loading inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve',
            run = False,
            load_only = True,
            coeff_masks = {101: mask,
                           103: mask,
                           501: reg_mask})

        print(f"-- Processing inversion parameter sensitivity...")
        diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir=f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/')

        fig, ax = pyissm.inversion.plot.plot_lcurve(diagnostics)
        ax.set_title('Grounded ice friction inversion - cf501 L-curve (p=q=1, warm-started)')
        plt.savefig(f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/lcurve.png')

    else:
        print(f"-- Running warm-started cf501 regularisation sweep...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve',
            run = True,
            load_only = False,
            coeff_masks = {101: mask,
                           103: mask,
                           501: reg_mask})